## IMDB Sentiment Analysis (DistilBERT)

This notebook trains a transformer model to classify IMDB movie reviews as **positive** or **negative**.


In [ ]:
import pandas as pd
import numpy as np
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


loading dataset

In [ ]:
import pandas as pd

DP = "/content/drive/MyDrive/IMDB Dataset.csv"

df = pd.read_csv(DP)
df.head()


## Exploratory Data Analysis

Checking class balance and review length to understand the dataset.


In [ ]:
df.info()


In [ ]:
import matplotlib.pyplot as plt

df['sentiment'].value_counts().plot(kind='bar', title='Sentiment Distribution')
plt.show()


In [ ]:
df['word_count'] = df['review'].apply(lambda x: len(str(x).split()))

df['word_count'].describe()


In [ ]:
plt.figure(figsize=(8,4))
plt.hist(df['word_count'], bins=50)
plt.title("Review Length Distribution")
plt.xlabel("Words")
plt.ylabel("Frequency")
plt.show()


In [ ]:
df["label"] = df["sentiment"].map({"negative": 0, "positive": 1})


## Text Preprocessing

Only minimal cleaning is applied because transformer models work better with raw text.


In [ ]:
def preprocess(text):
    text = str(text)
    text = text.replace("<br />", " ")
    text = text.strip()
    return text

df["new_rev"] = df["review"].apply(preprocess)


## Train–Test Split

Splitting the data into training and testing sets while keeping class balance.


In [ ]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["new_rev"],
    df["label"],
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)


In [ ]:
train_ds = Dataset.from_dict({
    "text": train_texts.tolist(),
    "label": train_labels.tolist()
})

test_ds = Dataset.from_dict({
    "text": test_texts.tolist(),
    "label": test_labels.tolist()
})


loading DistilBERT tokenizer from Hugging Face

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")


In [ ]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

train_ds = train_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)


removes the original text column and sets the dataset format to 'torch' tensors for model training

In [ ]:
train_ds = train_ds.remove_columns(["text"])
test_ds = test_ds.remove_columns(["text"])

train_ds.set_format("torch")
test_ds.set_format("torch")


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds)
    }


In [ ]:
training_args = TrainingArguments(
    output_dir="./distilbert_output",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    load_best_model_at_end=True,
    logging_steps=100,
    report_to="none"
)


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)


## Model Training

Fine-tuning DistilBERT on IMDB dataset for sentiment classification.


In [ ]:
trainer.train()


## Testing on New Input

Using the trained model to predict sentiment of new reviews.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_path = r'C:\vscode\CVprojects\NLP\sentimentModel\distilbert_imdb_model' # change if needed

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()




In [ ]:
import torch.nn.functional as F

def predict_sentiment(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    probs = F.softmax(outputs.logits, dim=1)
    pred = torch.argmax(probs).item()
    confidence = probs[0][pred].item()

    label = "POSITIVE" if pred == 1 else "NEGATIVE"
    return label, confidence


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
text = "i loved the movie but not now"
label, conf = predict_sentiment(text)

print("Prediction:", label)
print("Confidence:", round(conf, 3))